# memrust — PDF RAG with a LangGraph agent

The full pipeline, end to end:

1. load a real PDF with **pypdf** ("Attention Is All You Need") and chunk it
2. embed with **BGE-large-en-v1.5** (1024-dim) — or swap in OpenAI embeddings
3. store chunks + vectors in memrust
4. build a **LangGraph** retrieve→generate RAG agent over memrust
5. the agent **writes back**: reflections and episodic traces of what it did
6. the whole memory feature set: multi-agent private/shared visibility,
   working-memory TTLs, consolidation, entity graph, snapshots, checkpoints

In [ ]:
%pip install -q memrust pypdf sentence-transformers langgraph langchain-openai

In [ ]:
# Download the memrust server (one static Linux binary) and start it.
# Local machine? Build with `cargo build --release` and point BIN at
# target/release/memrust instead.
import os, subprocess, time, urllib.request

BIN = "./memrust-bin/memrust"
DATA_DIR = "./memory-pdf-rag"          # this notebook's own memory store

if not os.path.exists(BIN):
    os.makedirs("memrust-bin", exist_ok=True)
    url = ("https://github.com/AIAnytime/memrust/releases/download/"
           "v0.5.2/memrust-v0.5.2-x86_64-unknown-linux-musl.tar.gz")
    urllib.request.urlretrieve(url, "memrust-bin/memrust.tar.gz")
    subprocess.run(["tar", "xzf", "memrust.tar.gz"], cwd="memrust-bin", check=True)

# Re-running this cell (or another memrust notebook in the same runtime) can
# leave an old server holding port 7700 with a different data dir — replace it.
subprocess.run(["pkill", "-f", "memrust serve"], capture_output=True)
time.sleep(0.5)

server = subprocess.Popen(
    [BIN, "serve", "--data-dir", DATA_DIR,
     "--lifecycle-interval-secs", "0",     # we trigger lifecycle explicitly below
     "--consolidate-after-secs", "0"],     # so consolidation demos run immediately
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:7700/health", timeout=1)
        break
    except Exception:
        time.sleep(0.5)
print("memrust is up on http://127.0.0.1:7700 (data dir: memory-pdf-rag)")

## 1 · Load and chunk the PDF

In [ ]:
import urllib.request
from pypdf import PdfReader

urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", "attention.pdf")
reader = PdfReader("attention.pdf")
print(f"{len(reader.pages)} pages")

def chunk(text, size=800, overlap=120):
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i + size])
        i += size - overlap
    return out

chunks = []
for page_no, page in enumerate(reader.pages, start=1):
    for piece in chunk(" ".join((page.extract_text() or "").split())):
        if len(piece) > 100:
            chunks.append({"text": piece, "page": page_no})
print(f"{len(chunks)} chunks")

## 2 · Embed and store

sentence-transformers below. To use **OpenAI embeddings** instead, point the
server at them when you start it (then drop the `embedding=`/`query_embedding=`
arguments — the engine embeds server-side):

```bash
memrust serve --embedder openai --embedding-model text-embedding-3-small
```

For an asymmetric model served that way, memrust applies the prefixes for
you: `--embed-query-prefix "Represent this sentence for searching relevant
passages: "`.

In [ ]:
from memrust import MemrustClient
from sentence_transformers import SentenceTransformer

memory = MemrustClient("http://127.0.0.1:7700")
model = SentenceTransformer("BAAI/bge-large-en-v1.5")   # 1024-dim

# BGE is asymmetric: the instruction prefix goes on queries, not passages.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def emb(text):
    return model.encode(text, normalize_embeddings=True).tolist()

def emb_query(text):
    return model.encode(QUERY_PREFIX + text, normalize_embeddings=True).tolist()

vectors = model.encode([c["text"] for c in chunks], normalize_embeddings=True,
                       show_progress_bar=True).tolist()
stored = memory.remember_batch([
    {"text": c["text"], "kind": "semantic", "session_id": "paper",
     "embedding": v, "metadata": {"page": c["page"]}}
    for c, v in zip(chunks, vectors)
])
print(f"stored {len(stored)} chunks")
memory.health()

## 3 · Retrieval sanity check

In [ ]:
def retrieve(query, top_k=5):
    return memory.recall(query, query_embedding=emb_query(query),
                         top_k=top_k, session_id="paper")

for h in retrieve("how does multi-head attention work?", top_k=3):
    page = (h["record"].get("metadata") or {}).get("page", "?")
    print(f'{h["score"]:.4f}  p.{page}  {h["record"]["text"][:100]}')

## 4 · A LangGraph RAG agent

Two nodes: `retrieve` pulls grounded context from memrust, `generate` answers
with citations. Needs an OpenAI key.

In [ ]:
import getpass, os
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

class RagState(TypedDict):
    question: str
    context: str
    answer: str

def retrieve_node(state: RagState):
    hits = retrieve(state["question"], top_k=5)
    context = "\n".join(
        f"[p.{(h['record'].get('metadata') or {}).get('page', '?')}] {h['record']['text']}"
        for h in hits)
    return {"context": context}

llm = ChatOpenAI(model="gpt-4o-mini")

def generate_node(state: RagState):
    answer = llm.invoke(
        "Answer from the paper excerpts only; cite pages like [p.3].\n\n"
        f"Excerpts:\n{state['context']}\n\nQuestion: {state['question']}"
    ).content
    return {"answer": answer}

graph = StateGraph(RagState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)
rag_agent = graph.compile()

result = rag_agent.invoke({"question": "Why does the paper use scaled dot-product attention?"})
print(result["answer"])

## 5 · The agent writes back

Real agent memory is read *and write*. After each answer the agent stores an
episodic trace (what was asked) and can store reflections (what it learned
about its own process) — private to itself via `agent_id`.

In [ ]:
agent_memory = MemrustClient("http://127.0.0.1:7700", agent_id="rag-agent")

def answer_and_remember(question):
    result = rag_agent.invoke({"question": question})
    trace = f"Q: {question} -> answered from the paper"
    agent_memory.remember(trace, kind="episodic", session_id="qa-log",
                          embedding=emb(trace))
    return result["answer"]

print(answer_and_remember("What are the two sub-layers in each encoder layer?")[:300])
reflection = ("Reflection: positional-encoding questions need context from "
              "both section 3.5 and the appendix")
agent_memory.remember(reflection, kind="reflection", embedding=emb(reflection))

print()
print("agent's own episodic log:")
for h in agent_memory.recall("questions answered from the paper",
                             query_embedding=emb_query("questions answered from the paper"),
                             kinds=["episodic"]):
    print(" -", h["record"]["text"][:90])

## 6 · Multi-agent memory: private vs shared

Agent-owned memories default **private**. Mark them `shared` to publish to the
team; recall as an agent sees shared + unowned + its own.

In [ ]:
researcher = MemrustClient("http://127.0.0.1:7700", agent_id="researcher")
writer     = MemrustClient("http://127.0.0.1:7700", agent_id="writer")

NOTES = "research-notes"     # keep the demo clear of the 65 paper chunks

draft = "Draft thesis: attention scales better than recurrence for long sequences"
researcher.remember(draft, kind="working", session_id=NOTES,
                    embedding=emb(draft))                            # private
finding = "Finding: the paper reports 28.4 BLEU on WMT14 EN-DE"
researcher.remember(finding, kind="semantic", visibility="shared",
                    session_id=NOTES, embedding=emb(finding))        # shared

def visible(client, label):
    hits = client.recall("thesis findings BLEU", session_id=NOTES,
                         query_embedding=emb_query("thesis findings BLEU"), top_k=5)
    print(f"{label} sees {len(hits)}: {[h['record']['text'][:44] for h in hits]}")

visible(writer, "writer    ")       # 1 — only the shared finding
visible(researcher, "researcher")   # 2 — its own private draft too

## 7 · Entities, lifecycle, snapshots

In [ ]:
# The entity graph built itself during ingest — explore it.
import urllib.request, json as _json
ents = _json.load(urllib.request.urlopen("http://127.0.0.1:7700/v1/entities?limit=10"))
print("top entities:", [(e["name"], e["count"]) for e in ents["entities"]][:8])

In [ ]:
import time

# Working memory expires...
writer.remember("scratch: outline for the summary post", kind="working",
                ttl_seconds=3, embedding=emb("scratch: outline for the summary post"))
time.sleep(4)

# ...and the lifecycle pass sweeps it AND consolidates the agent's episodic
# q&a log into a semantic summary (server runs with --consolidate-after-secs 0).
for i in range(3):
    note = f"Q: follow-up {i} about attention heads -> answered"
    agent_memory.remember(note, kind="episodic", session_id="qa-log",
                          embedding=emb(note))
report = memory.run_lifecycle()
print("lifecycle report:", report)
if report["summaries"]:
    # Summaries preserve provenance and inherit the strictest visibility.
    print("qa-log consolidated into", len(report["summaries"]), "summary(ies)")

In [ ]:
snap = memory.snapshot("paper")
print(f"snapshot of the paper session: {len(snap['records'])} records "
      "(restorable on any memrust instance, ids preserved)")
memory.checkpoint()
print("final state:", memory.health())

## Wrap-up

One engine handled everything a five-system stack usually does: vector store,
keyword index, entity graph, reranking fusion, agent memory semantics,
lifecycle management, and multi-agent access control — behind three verbs.

- Repo: https://github.com/AIAnytime/memrust
- MCP for Claude Code / Desktop: `memrust mcp --agent-id <name>`

Open the dashboard to browse everything this notebook stored:

In [ ]:
# The dashboard is served by the engine itself at http://127.0.0.1:7700/.
# In Colab the server runs inside the VM, so that address in *your* browser
# points at your own machine and will refuse the connection — proxy the
# kernel's port instead:
try:
    from google.colab import output
    output.serve_kernel_port_as_window(7700)     # opens the dashboard in a tab
except ImportError:
    print("Not in Colab — open http://127.0.0.1:7700/ in your browser")